# 08 - Defect Classification

**Purpose:** Classify the defect type after anomaly detection.

**Expected output:** Class distribution, classifier method, prediction probabilities, classification report, and confusion matrix.

**Platform connection:** The backend uses this output to show `good`, `broken_large`, `broken_small`, or `contamination`.


In [1]:
from pathlib import Path
import json
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BOTTLE_DIR = PROJECT_ROOT / "data" / "raw" / "mvtec_anomaly_detection" / "bottle"
IMAGE_SIZE = (256, 256)
LABELS = ["good", "broken_large", "broken_small", "contamination"]


def read_rgb(path):
    image_bgr = cv2.imread(str(path))
    if image_bgr is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def first_image(label="contamination", split="test"):
    folder = BOTTLE_DIR / split / label
    images = sorted(folder.glob("*.png"))
    if not images:
        raise FileNotFoundError(f"No images found in {folder}")
    return images[0]


def show_grid(items, cols=4, figsize=(14, 7), suptitle=None):
    rows = int(np.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.atleast_1d(axes).ravel()
    for ax, item in zip(axes, items):
        title, image, cmap = item
        ax.imshow(image, cmap=cmap)
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.axis("off")
    for ax in axes[len(items):]:
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()


print("Project root:", PROJECT_ROOT)
print("MVTec bottle folder exists:", BOTTLE_DIR.exists())

from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from ml.dataset_loader import load_bottle_dataframe
from ml.classifier import LABEL_ORDER, load_classifier_bundle, build_resnet18_feature_extractor, extract_features
MODEL_PATH = PROJECT_ROOT / "models" / "defect_classifier.pkl"


Project root: C:\Users\HP\Desktop\springboard\visioninspect-ai-github
MVTec bottle folder exists: True


## Class Distribution


In [ ]:
dataset_df = load_bottle_dataframe()
classification_df = dataset_df[(dataset_df["split"].eq("test")) & (dataset_df["label"].isin(LABEL_ORDER))].copy().reset_index(drop=True)
class_counts = classification_df.groupby("label").size().reindex(LABEL_ORDER, fill_value=0).reset_index(name="count")
display(class_counts)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(class_counts["label"], class_counts["count"], color=["#16a34a", "#dc2626", "#f59e0b", "#7c3aed"])
ax.set_title("Defect Classification Class Distribution")
ax.set_ylabel("Images")
plt.tight_layout()
plt.show()


## Method And Artifact


In [ ]:
display(pd.DataFrame([
    {"step": 1, "component": "ResNet18 feature extractor", "purpose": "convert image to pretrained visual embedding"},
    {"step": 2, "component": "StandardScaler", "purpose": "normalize embedding features"},
    {"step": 3, "component": "Logistic Regression", "purpose": "classify defect type from embeddings"},
    {"step": 4, "component": "Probability output", "purpose": "provide confidence for severity scoring"},
]))
display(pd.DataFrame([{"classifier_path": str(MODEL_PATH), "exists": MODEL_PATH.exists(), "size_kb": round(MODEL_PATH.stat().st_size / 1024, 2) if MODEL_PATH.exists() else None}]))


## Evaluate Saved Classifier


In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Classifier artifact missing: {MODEL_PATH}")
bundle = load_classifier_bundle(MODEL_PATH)
feature_extractor, preprocess, device = build_resnet18_feature_extractor()
features = extract_features(classification_df["image_path"].tolist(), batch_size=16, feature_extractor=feature_extractor, preprocess=preprocess, device=device)
classifier = bundle["classifier"]
y_true = classification_df["label"].tolist()
y_pred = classifier.predict(features).tolist()
probabilities = classifier.predict_proba(features)
classes = classifier.classes_.tolist()
labels = [label for label in LABEL_ORDER if label in classes]
report_df = pd.DataFrame(classification_report(y_true, y_pred, labels=labels, zero_division=0, output_dict=True)).transpose().reset_index().rename(columns={"index": "label"})
display(report_df)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=labels, cmap="Blues", xticks_rotation=25, ax=ax)
ax.set_title("Defect Classifier Confusion Matrix")
plt.tight_layout()
plt.show()


## Probability Examples


In [ ]:
rows = []
for idx in range(min(8, len(classification_df))):
    row = classification_df.iloc[idx]
    probs = {f"prob_{cls}": round(float(prob), 4) for cls, prob in zip(classes, probabilities[idx], strict=False)}
    rows.append({"image": Path(row["image_path"]).name, "true_label": row["label"], "predicted_label": y_pred[idx], "confidence": round(float(np.max(probabilities[idx])), 4), **probs})
display(pd.DataFrame(rows))
items = [(f"True: {classification_df.iloc[idx]['label']}\nPred: {y_pred[idx]}", read_rgb(classification_df.iloc[idx]["image_path"]), None) for idx in range(min(4, len(classification_df)))]
show_grid(items, cols=4, figsize=(14, 4), suptitle="Classifier Prediction Examples")


## Classifier Error And Confidence Analysis

This shows whether mistakes are low-confidence and therefore suitable for manual review.


In [ ]:
analysis_df = pd.DataFrame({
    "true_label": y_true,
    "predicted_label": y_pred,
    "confidence": np.max(probabilities, axis=1),
})
analysis_df["correct"] = analysis_df["true_label"] == analysis_df["predicted_label"]
summary = analysis_df.groupby("correct")["confidence"].agg(["count", "mean", "min", "max"]).reset_index()
display(summary)

fig, ax = plt.subplots(figsize=(7, 4))
for correct, group in analysis_df.groupby("correct"):
    ax.hist(group["confidence"], bins=10, alpha=0.65, label="correct" if correct else "incorrect")
ax.set_title("Classifier Confidence Distribution")
ax.set_xlabel("Confidence")
ax.set_ylabel("Images")
ax.legend()
plt.tight_layout()
plt.show()

low_confidence = analysis_df.sort_values("confidence").head(8)
display(low_confidence)


## Validation Outcome

- Defect classification is separate from anomaly detection.
- The classifier returns probabilities for confidence and severity scoring.
- The artifact can be loaded and evaluated without overwriting production model files.
